# K-Visitation worked example

The sample in `data/sample/` contains the toy user's stays and destination attributes. Its visit counts, home distances and amenity categories follow the Supplementary Information example.

`sample_places.csv` contains synthetic place attributes. Coordinates are generated around a fictional home at (385000, 6670000) metres in EPSG:3067, using bearings clockwise from grid north. Euclidean home distances in that CRS match `home_dist_km`; `lat` and `lng` are stored in WGS84 (EPSG:4326).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'src' / 'k_visitation.py').is_file())
sys.path.insert(0, str(root / 'src'))

from k_visitation import calculate_k_visitation
from prepare_visits import aggregate_visits

## 1. Sample stays and POI exposure

In [ ]:
sample_dir = root / 'data' / 'sample'
stays = pd.read_csv(sample_dir / 'sample_stays.csv', parse_dates=['start', 'end'])
sample_places = pd.read_csv(sample_dir / 'sample_places.csv')
display(stays.head(), sample_places)

## 2. Visitation and amenity coverage

Visit frequency is the number of stays at each location. The location attributes provide the distance from home and the amenity categories present at each destination.

In [ ]:
visits = aggregate_visits(stays.assign(visit_freq=1), ['user_id', 'cell_id'], sort=True)
locations = visits.merge(sample_places, on='cell_id', validate='many_to_one')
display(locations)

## 3. K-Visitation and $q_K$

$K_{freq}$ adds locations from most to least visited, and $K_{dist}$ adds them from nearest to farthest from home. Each set stops growing once it covers every amenity category in the user's visited locations. Equal ranks retain the order in the visitation table.

The alignment score $q_K$ is the number of visits to locations shared by both sets divided by the number of visits to locations in either set.

In [ ]:
places, user_qk = calculate_k_visitation(locations, distance_col='home_dist_km')
k_freq = (places.loc[places['k_freq'].eq(1)]
          .sort_values('visit_freq', ascending=False, kind='stable')['cell_id'].tolist())
k_dist = (places.loc[places['k_dist'].eq(1)]
          .sort_values('home_dist_km', kind='stable')['cell_id'].tolist())

print('K_freq:', ', '.join(k_freq))
print('K_dist:', ', '.join(k_dist))

result = user_qk.iloc[0]
intersection_visits = int(result['intersection_visits'])
union_visits = int(result['union_visits'])
print(f"q_K = {intersection_visits}/{union_visits} = {result['qk_weighted']:.6f}")